In [20]:
import time
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.feature_selection import (
    SelectFromModel, SelectKBest, f_classif, f_regression, RFE
)
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.svm import SVC, LinearSVC, SVR, LinearSVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, roc_auc_score,
    classification_report, mean_absolute_error, mean_squared_error, r2_score
)

In [2]:
DATASET_PATH = os.path.join('..', 'Pediatric_Anxiety_Disorder')
PHENOTYPE_PATH = os.path.join(DATASET_PATH, 'phenotype', 'phenotype_preprocessed.tsv')
FEATURE_TABLE_PATH = os.path.join(
    '..', 'dataset_holdouts', '10_holdouts', 'holdout-09', 
    'uncorrected_alpha-0.01_two-sided_cluster-thresh-20', 
    'ML_Dataset_holdout-09.csv'
)

In [3]:
phenotype = pd.read_csv(PHENOTYPE_PATH, sep='\t')
phenotype.head()

,participant_id,sex,age_baseline,KSADS_MAIN_DIAGNOSIS,WASI_FULL_2_IQ,INCOME,RACE_WHITE,RACE_BLACK,RACE_ASIAN,RACE_MULTIPLE,ETHNICITY,COHORT,SCANNER,SCARED_BASELINE_YOUTH_RATING,SCARED_BASELINE_PARENT_RATING,CGAS_BASELINE_SCORE,PARS_BASELINE_CLINICIAN_RATING,IQ_IS_MISSING,INCOME_IS_MISSING
0,sub-020131,1.0,17.0,HV,122.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,4.0,7.2,NaN,NaN,-1,-1
1,sub-020209,1.0,16.0,HV,118.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,23.0,17.0,NaN,NaN,-1,-1
2,sub-020725,1.0,14.0,HV,116.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,3.0,0.0,NaN,NaN,-1,-1
3,sub-021100,-1.0,11.0,HV,134.0,8.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,8.0,1.0,NaN,NaN,-1,-1
4,sub-021214,1.0,11.0,HV,130.0,9.0,1.0,-1.0,-1.0,-1.0,-1.0,1,1,16.0,1.0,NaN,NaN,-1,-1


In [4]:
feature_table = pd.read_csv(FEATURE_TABLE_PATH)
feature_table.head()

,participant_id,LH_VisCent_Striate_1,LH_VisCent_Striate_2,LH_VisCent_Striate_3,LH_VisCent_Striate_4,LH_VisCent_ExStr_5,LH_VisCent_ExStr_7,LH_VisCent_ExStr_8,LH_VisCent_ExStr_9,LH_VisCent_ExStr_13,...,RH_DefaultA_PFCm_8_to_RH_DefaultB_Temp_1,RH_DefaultA_PFCm_8_to_RH_DefaultB_PFCd_1,RH_DefaultA_PFCm_8_to_RH_DefaultB_PFCd_8,RH_DefaultA_PFCm_8_to_RH_DefaultC_Rsp_3,RH_DefaultB_Temp_1_to_RH_DefaultB_PFCd_1,RH_DefaultB_Temp_1_to_RH_DefaultB_PFCd_8,RH_DefaultB_Temp_1_to_RH_DefaultC_Rsp_3,RH_DefaultB_PFCd_1_to_RH_DefaultB_PFCd_8,RH_DefaultB_PFCd_1_to_RH_DefaultC_Rsp_3,RH_DefaultB_PFCd_8_to_RH_DefaultC_Rsp_3
0,sub-022818,-0.792813,-0.209112,-0.921329,-0.074651,-0.552897,-0.322209,-0.536953,-0.083299,-0.061983,...,0.009694,0.119019,-0.116757,-0.089229,-0.036792,-0.042153,0.143288,-0.047351,0.085100,-0.135077
1,sub-022829,0.250197,0.344616,0.125904,0.179377,-0.350762,-0.185640,-0.206080,-0.027430,0.172566,...,0.115506,0.019921,0.081861,0.080610,-0.053288,0.016046,-0.068698,0.224956,0.063904,0.121332
2,sub-022831,0.183666,0.013184,0.434617,0.000000,0.992951,-0.236570,0.280258,-0.194933,-0.009808,...,0.066555,0.181309,0.073581,-0.085020,0.026873,-0.029152,0.062724,0.039342,-0.054859,-0.024381
3,sub-022847,0.056499,0.025390,0.307906,0.000000,0.906192,0.380258,0.232378,0.028523,0.010135,...,0.042978,0.077684,0.050804,0.004865,0.166685,0.054788,0.049129,0.035331,0.085572,-0.013684
4,sub-022871,0.903973,0.818287,1.108589,0.320412,0.663097,0.572853,0.771676,0.596555,0.354556,...,0.022551,0.197140,0.046102,0.041622,0.388080,-0.073552,0.047793,0.037226,-0.093330,-0.005227


In [5]:
# fetching the target columns
data = pd.merge(
    feature_table,
    phenotype[['participant_id', 'KSADS_MAIN_DIAGNOSIS', 'SCARED_BASELINE_YOUTH_RATING']],
    on='participant_id',
    how='inner'
).rename(
    columns={
        'KSADS_MAIN_DIAGNOSIS': 'anxiety',
        'SCARED_BASELINE_YOUTH_RATING': 'scared_score'
    }
)
data['anxiety'] = data['anxiety'].map({'HV':0, 'ANX':1})
data.head()

,participant_id,LH_VisCent_Striate_1,LH_VisCent_Striate_2,LH_VisCent_Striate_3,LH_VisCent_Striate_4,LH_VisCent_ExStr_5,LH_VisCent_ExStr_7,LH_VisCent_ExStr_8,LH_VisCent_ExStr_9,LH_VisCent_ExStr_13,...,RH_DefaultA_PFCm_8_to_RH_DefaultB_PFCd_8,RH_DefaultA_PFCm_8_to_RH_DefaultC_Rsp_3,RH_DefaultB_Temp_1_to_RH_DefaultB_PFCd_1,RH_DefaultB_Temp_1_to_RH_DefaultB_PFCd_8,RH_DefaultB_Temp_1_to_RH_DefaultC_Rsp_3,RH_DefaultB_PFCd_1_to_RH_DefaultB_PFCd_8,RH_DefaultB_PFCd_1_to_RH_DefaultC_Rsp_3,RH_DefaultB_PFCd_8_to_RH_DefaultC_Rsp_3,anxiety,scared_score
0,sub-022818,-0.792813,-0.209112,-0.921329,-0.074651,-0.552897,-0.322209,-0.536953,-0.083299,-0.061983,...,-0.116757,-0.089229,-0.036792,-0.042153,0.143288,-0.047351,0.085100,-0.135077,1,29.0
1,sub-022829,0.250197,0.344616,0.125904,0.179377,-0.350762,-0.185640,-0.206080,-0.027430,0.172566,...,0.081861,0.080610,-0.053288,0.016046,-0.068698,0.224956,0.063904,0.121332,1,34.0
2,sub-022831,0.183666,0.013184,0.434617,0.000000,0.992951,-0.236570,0.280258,-0.194933,-0.009808,...,0.073581,-0.085020,0.026873,-0.029152,0.062724,0.039342,-0.054859,-0.024381,1,12.0
3,sub-022847,0.056499,0.025390,0.307906,0.000000,0.906192,0.380258,0.232378,0.028523,0.010135,...,0.050804,0.004865,0.166685,0.054788,0.049129,0.035331,0.085572,-0.013684,1,18.0
4,sub-022871,0.903973,0.818287,1.108589,0.320412,0.663097,0.572853,0.771676,0.596555,0.354556,...,0.046102,0.041622,0.388080,-0.073552,0.047793,0.037226,-0.093330,-0.005227,0,6.0


In [6]:
# defining features and targets
feature_cols = [c for c in data.columns
               if c not in ['participant_id', 'anxiety', 'scared_score']]
X = data[feature_cols].values
y_diagnosis = data['anxiety'].values
y_severity = data['scared_score'].values

In [7]:
print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution: {pd.Series(y_diagnosis).value_counts().to_dict()}")

Feature matrix shape: (89, 4278)
Class distribution: {1: 46, 0: 43}


In [9]:
# Leave One Out Cross-validation
loo = LeaveOneOut()

## CLASSIFICATION - Anxiety vs Healthy

In [18]:
classification_pipelines = {
    'LogisticElasticNet': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            penalty='elasticnet', l1_ratio=0.7, C=0.1, 
            solver='saga', max_iter=5000, random_state=42
        ))
    ]),
    'LogisticRFE': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', RFE(
            estimator=LogisticRegression(
                l1_ratio=0, C=0.1, solver='lbfgs', # helps score the features but l2 doesn't remove anything so no premature elimination of features
                max_iter=5000, random_state=42
            ),
            n_features_to_select=20,
            step=0.1
        )),
        ('classifier', LogisticRegression(
            penalty='elasticnet', l1_ratio=0.7, C=0.1, # double feature selection
            solver='saga', max_iter=5000, random_state=42
        ))
    ]),
    'SVM_Univariate': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(f_classif, k=20)),
        ('classifier', SVC(
            kernel='linear', C=1.0, probability=True, random_state=42
        ))
    ]),
    'SVM_ElasticNet': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectFromModel(
            LogisticRegression(
            penalty='elasticnet', l1_ratio=0.7, C=0.1, 
            solver='saga', max_iter=5000, random_state=42
        )
        )),
        ('classifier', SVC(
            kernel='linear', C=1.0, probability=True, random_state=42
        ))
    ]),
    'SVM_RFE': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', RFE(
            estimator=LinearSVC(C=0.1, max_iter=5000, random_state=42),
            n_features_to_select=20,
            step=0.1
        )),
        ('classifier', SVC(
            kernel='linear', C=1.0, probability=True, random_state=42
        ))
    ]),
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(
            n_estimators=100, random_state=42
        ))
    ]),
    'RandomForestRFE': Pipeline([
        ('scaler', StandardScaler()),
        ('selector', RFE(
            estimator=RandomForestClassifier(
                n_estimators=100, random_state=42
            ),
            n_features_to_select=20,
            step=0.1
        )),
        ('classifier', RandomForestClassifier(
            n_estimators=100, random_state=42
        ))
    ])
}

In [22]:
clf_results = {}
for name, pipeline in classification_pipelines.items():
    print(f"\nRunning {name}...")
    start_time = time.time()

    y_pred = cross_val_predict(pipeline, X, y_diagnosis, cv=loo)
    y_prob = cross_val_predict(pipeline, X, y_diagnosis, cv=loo, method='predict_proba')[:, 1]

    end_time = time.time()
    duration = end_time - start_time
    acc = accuracy_score(y_diagnosis, y_pred)
    bal_acc = balanced_accuracy_score(y_diagnosis, y_pred)
    auc = roc_auc_score(y_diagnosis, y_prob)

    clf_results[name] = {
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'auc_roc': auc,
        'duration_sec': duration
    }

    print(f"  Duration:          {duration:.3f}s")
    print(f"  Accuracy:          {acc:.3f}")
    print(f"  Balanced Accuracy: {bal_acc:.3f}")
    print(f"  AUC-ROC:           {auc:.3f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_diagnosis, y_pred))

clf_summary = pd.DataFrame(clf_results).T
print("\n --- Classification Summary ---")
print(clf_summary.round(3))


Running LogisticElasticNet...
  Duration:          1133.852s
  Accuracy:          0.360
  Balanced Accuracy: 0.356
  AUC-ROC:           0.271

  Classification Report:
              precision    recall  f1-score   support

           0       0.31      0.26      0.28        43
           1       0.40      0.46      0.42        46

    accuracy                           0.36        89
   macro avg       0.35      0.36      0.35        89
weighted avg       0.35      0.36      0.35        89


Running LogisticRFE...


C:\Users\aleen\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
C:\Users\aleen\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
C:\Users\aleen\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
C:\Users\aleen\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
C:\Users\aleen\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221:

  Duration:          33.232s
  Accuracy:          0.371
  Balanced Accuracy: 0.368
  AUC-ROC:           0.279

  Classification Report:
              precision    recall  f1-score   support

           0       0.32      0.28      0.30        43
           1       0.40      0.46      0.43        46

    accuracy                           0.37        89
   macro avg       0.36      0.37      0.36        89
weighted avg       0.37      0.37      0.37        89


Running SVM_Univariate...
  Duration:          4.592s
  Accuracy:          0.607
  Balanced Accuracy: 0.607
  AUC-ROC:           0.658

  Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.60      0.60        43
           1       0.62      0.61      0.62        46

    accuracy                           0.61        89
   macro avg       0.61      0.61      0.61        89
weighted avg       0.61      0.61      0.61        89


Running SVM_ElasticNet...
  Duration:          